# NDWI Analysis: Identifying Actively Irrigated Center Pivot Systems

## Purpose
Classify center pivot irrigation systems (CPIS) in arid Sub-Saharan Africa as actively
irrigated or dormant/abandoned using the Normalized Difference Water Index (NDWI)
derived from Sentinel-2 satellite imagery.

## Context
This notebook is **Step 2** in the water source attribution arc:

1. *CPIS expansion* — Notebooks in `Code/1_analyze_data` established that CPIS expanded
   191% in SSA between 2000 and 2021.
2. **Which systems are actually active? ← This notebook**
   Chen et al. (2023) mapped every CPIS-shaped polygon, but some are fallow or
   abandoned. NDWI separates actively irrigated systems from inactive ones, providing
   a cleaned denominator for all downstream water-source analyses. Analyzing water
   source relationships for abandoned systems would conflate two distinct phenomena.
3. *Dam accessibility* — `6_dem_flow_analysis.ipynb` refines distance-to-dam analysis
   using elevation, establishing which active CPIS could plausibly use surface water.
4. *Groundwater* — `7_groundwater_gp_regression.ipynb` tests whether active CPIS
   correlate with groundwater availability.
6. *Anomalies* — `6_anomaly_detection.ipynb` isolates systems unexplained by either source.

NDWI = (Green − NIR) / (Green + NIR)
Positive values indicate open water or actively irrigated crops; negative values
indicate bare soil or dry vegetation.

## Inputs
| Dataset | Config key | Notes |
|---------|-----------|-------|
| Sentinel-2 NDWI tiles (pre-computed in GEE) | `Sentinel2_dir_path` | **Must be downloaded — see Data Acquisition below** |
| CPIS polygons (arid SSA, combined) | `SSA_Combined_CPIS_All_shp_path` | From `0_process_data` |
| Africa boundaries | `Africa_boundaries_shp_path` | For visualization |
| Arid SSA regions | `SSA_All_by_Country_shp_path` | For visualization |

## Outputs
| File | Config key | Description |
|------|-----------|-------------|
| `ndwi_mosaic.vrt` | *(in `Sentinel2_dir_path`)* | VRT mosaic of GEE-exported NDWI tiles |
| `Active_CPIS.shp` | `Active_CPIS_shp_path` | CPIS with mean NDWI > 0 |
| `Inactive_CPIS.shp` | `Inactive_CPIS_shp_path` | CPIS with mean NDWI ≤ 0 |
| `CPIS_NDWI_Stats.csv` | `CPIS_NDWI_stats_csv_path` | Per-polygon NDWI statistics |

## Data Acquisition: Obtaining Sentinel-2 Imagery

Sentinel-2 Level-2A surface reflectance imagery is required but not included in the
repository.

### Why max NDWI, not median?
In arid SSA, irrigated center pivots are only green for a fraction of the year.
A **median** annual composite is dominated by dry/fallow periods and returns negative
NDWI even for actively-used fields. A **90th-percentile** composite captures peak
greenness: if a field was irrigated at any point during the year, it will show up.

### Why compute NDWI in GEE, not Python?
NDWI = (Green − NIR) / (Green + NIR) should be computed **per image before
aggregating**, not from aggregated bands. Exporting pre-computed NDWI also halves
download size (one file instead of two band sets).

### Google Earth Engine export script
Run at https://code.earthengine.google.com

```javascript
// ============================================================
// GEE Export: 90th-percentile annual NDWI for arid SSA (2021)
// ============================================================
var aridSSA = ee.FeatureCollection('users/YOUR_USERNAME/SSA_All_Arid_by_Country');
// Upload SSA_All_Arid_by_Country.shp to your GEE assets first

// Compute NDWI per image, then take the 90th percentile across the year.
// This captures peak greenness while being robust to cloud artifacts.
var ndwi = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
  .filterBounds(aridSSA)
  .filterDate('2021-01-01', '2021-12-31')
  .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
  .map(function(img) {
    return img.normalizedDifference(['B3', 'B8A']).rename('NDWI')
              .copyProperties(img, ['system:time_start']);
  })
  .reduce(ee.Reducer.percentile([90]))
  .rename('NDWI')
  .clip(aridSSA);

Export.image.toDrive({
  image: ndwi,
  description: 'Sentinel2_NDWI_AridSSA_2021_p90',
  folder: 'Africa_Irrigation',
  region: aridSSA.geometry().bounds(),
  scale: 60,
  crs: 'EPSG:4326',
  maxPixels: 1e13
});
```

**Place downloaded tile(s) at:**
```
Data/Raw/Sentinel2/Sentinel2_NDWI_AridSSA_2021_p90*.tif
```
GEE may split large exports into multiple tiles (e.g. `...-0000000000-0000000000.tif`);
place them all in the same folder — the check cell below will mosaic them automatically.

In [1]:
# --- Import Required Libraries and Utilities ---
import os
import sys
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import rasterio.mask
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import TwoSlopeNorm
from shapely.geometry import mapping

project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from Code.utils.utility import load_config, resolve_path

config = load_config()

## Check Data Availability

In [2]:
# --- Check for GEE-exported NDVI tiles, building VRT mosaic if needed ---
import glob as _glob
from osgeo import gdal as _gdal

sentinel2_dir = resolve_path(config['Sentinel2_dir_path'])
ndvi_vrt_path = os.path.join(sentinel2_dir, 'ndvi_mosaic.vrt')

def _build_vrt(pattern, out_path, label):
    tiles = sorted(_glob.glob(pattern))
    if not tiles:
        return False
    print(f"Building VRT for {len(tiles)} {label} tile(s) ...")
    vrt = _gdal.BuildVRT(out_path, tiles)
    vrt.FlushCache()
    vrt = None
    print(f"  Done: {os.path.basename(out_path)}")
    return True

if not os.path.isfile(ndvi_vrt_path):
    _build_vrt(
        os.path.join(sentinel2_dir, 'Sentinel2_NDVI_AridSSA_2021_p90*.tif'),
        ndvi_vrt_path,
        'NDVI',
    )

DATA_AVAILABLE = os.path.isfile(ndvi_vrt_path)

if not DATA_AVAILABLE:
    print("WARNING: Sentinel-2 NDVI data not found.")
    print(f"  Expected tiles matching:")
    print(f"  {os.path.join(sentinel2_dir, 'Sentinel2_NDVI_AridSSA_2021_p90*.tif')}")
    print("See the 'Data Acquisition' section above for GEE export instructions.")
    print("Analysis cells below will skip until data is available.")
else:
    print("Sentinel-2 NDVI data found. Proceeding with analysis.")

Sentinel-2 NDVI data found. Proceeding with analysis.


## Prepare NDWI Mosaic

NDWI = (Green − NIR) / (Green + NIR) was computed **per image** in Google Earth Engine
before aggregating to a 90th-percentile annual composite (see Data Acquisition above).
This cell validates the VRT mosaic built from the downloaded tiles and records the CRS
for downstream cells.

Values range from −1 to +1:
- **Positive (> 0):** Open water or actively irrigated vegetation
- **Near zero (−0.1 to 0):** Sparse or stressed vegetation
- **Negative (< −0.1):** Bare soil, dry vegetation, or built surfaces

For CPIS activity classification, NDWI > 0 is used as the threshold for active
irrigation. This can be tuned based on regional crop calendars and validation data.

In [3]:
# --- Validate GEE-exported NDVI mosaic and record CRS ---
if DATA_AVAILABLE:
    with rasterio.open(ndvi_vrt_path) as _src:
        crs_s2 = _src.crs
        print(f"NDVI mosaic ready.")
        print(f"  Path : {ndvi_vrt_path}")
        print(f"  CRS  : {crs_s2}")
        print(f"  Size : {_src.width} x {_src.height} pixels")
        print(f"  Bounds: {_src.bounds}")
else:
    print("[Skipped] Sentinel-2 data required. See Data Acquisition section.")

NDVI mosaic ready.
  Path : C:/Users/ermil/Documents/Africa_Irrigation\Data/Raw/Sentinel2\ndvi_mosaic.vrt
  CRS  : EPSG:4326
  Size : 127939 x 115275 pixels
  Bounds: BoundingBox(left=-17.54194154217237, bottom=-34.83379210924586, right=51.41579393880812, top=27.298184516880845)


## Load CPIS Polygons

We use the combined (2000 + 2021) CPIS dataset for arid SSA. Each polygon represents
one center pivot system observed in either year.

In [4]:
# --- Load and reproject CPIS polygons ---
cpis = gpd.read_file(resolve_path(config['SSA_Combined_CPIS_All_shp_path']))

if DATA_AVAILABLE:
    with rasterio.open(ndvi_vrt_path) as _src:
        cpis = cpis.to_crs(_src.crs)

print(f"CPIS polygons loaded: {len(cpis)}")
print(f"CRS: {cpis.crs}")
print(f"Columns: {list(cpis.columns)}")

CPIS polygons loaded: 29482
CRS: GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST],AUTHORITY["EPSG","4326"]]
Columns: ['ID', 'Country', 'Country Co', 'Year', 'geometry']


## Extract NDWI Statistics per CPIS Polygon

For each CPIS polygon we extract all Sentinel-2 pixels within its boundary and
compute summary statistics. Mean NDWI drives the active/inactive classification;
standard deviation captures within-pivot variability.

In [5]:
# --- Extract per-polygon NDVI statistics ---
if DATA_AVAILABLE:
    from rasterstats import zonal_stats

    stats_path = resolve_path(config['CPIS_NDWI_stats_csv_path'])

    if os.path.isfile(stats_path):
        print(f"NDVI stats already exist, loading from disk.")
        print(f"  {stats_path}")
        print("  Delete this file to re-extract.")
        ndvi_df = pd.read_csv(stats_path, index_col='cpis_idx')
        cpis = cpis.join(ndvi_df)
        valid_count = cpis['ndvi_mean'].notna().sum()
        print(f"Loaded stats for {valid_count:,} / {len(cpis):,} CPIS polygons")
    else:
        print(f"Extracting NDVI for {len(cpis):,} polygons via zonal_stats ...")
        stats = zonal_stats(
            cpis,
            ndvi_vrt_path,
            stats=['mean', 'median', 'std', 'count'],
            nodata=np.nan,
            all_touched=True,
        )

        ndvi_df = pd.DataFrame([{
            'cpis_idx': idx,
            'ndvi_mean':   s.get('mean'),
            'ndvi_median': s.get('median'),
            'ndvi_std':    s.get('std'),
            'n_pixels':    s.get('count', 0),
        } for idx, s in zip(cpis.index, stats)]).set_index('cpis_idx')

        cpis = cpis.join(ndvi_df)
        ndvi_df.to_csv(stats_path)

        valid_count = cpis['ndvi_mean'].notna().sum()
        print(f"Done. NDVI extracted for {valid_count:,} / {len(cpis):,} CPIS polygons")
        print(f"Saved to: {stats_path}")

    print(cpis[['ndvi_mean', 'ndvi_std', 'n_pixels']].describe().round(3))
else:
    print("[Skipped] Requires NDVI mosaic from previous cell.")

NDVI stats already exist, loading from disk.
  C:/Users/ermil/Documents/Africa_Irrigation\Data/Processed/CPIS_NDWI_Stats.csv
  Delete this file to re-extract.
Loaded stats for 29,482 / 29,482 CPIS polygons
       ndvi_mean   ndvi_std   n_pixels
count  29482.000  29482.000  29482.000
mean       0.654      0.106    102.666
std        0.182      0.049     67.983
min       -0.187      0.005      2.000
25%        0.509      0.068     56.000
50%        0.702      0.101     83.000
75%        0.811      0.138    130.000
max        0.956      0.465   1460.000


## Classify CPIS as Active or Inactive

**Threshold:** mean NDVI > 0.2 → actively irrigated

NDVI = (NIR − Red) / (NIR + Red):
- Irrigated crops (green vegetation) → high NIR, moderate Red → **NDVI 0.3–0.7**
- Bare/fallow arid soil → low NIR and Red → **NDVI near 0**
- Open water → low NIR → **NDVI negative**

The 90th-percentile composite captures peak greenness across 2021, so any field
that was irrigated at any point in the year will show elevated NDVI.

Adjust `NDVI_THRESHOLD` based on validation data. 0.2 is a standard lower bound
for actively vegetated land in dryland agriculture literature.

In [6]:
# --- Classify and save active/inactive CPIS ---
if DATA_AVAILABLE:
    NDVI_THRESHOLD = 0.2  # active if peak-year NDVI > this

    active_path   = resolve_path(config['Active_CPIS_shp_path'])
    inactive_path = resolve_path(config['Inactive_CPIS_shp_path'])
    stats_path    = resolve_path(config['CPIS_NDWI_stats_csv_path'])

    if os.path.isfile(active_path) and os.path.isfile(inactive_path):
        print("Active/inactive shapefiles already exist, skipping classification.")
        print(f"  {active_path}")
        print(f"  {inactive_path}")
        print("  Delete these files to re-classify (e.g. with a different NDVI_THRESHOLD).")
    else:
        cpis['active'] = cpis['ndvi_mean'] > NDVI_THRESHOLD

        active_cpis   = cpis[cpis['active'] == True].copy()
        inactive_cpis = cpis[cpis['active'] == False].copy()

        n_total    = len(cpis)
        n_active   = len(active_cpis)
        n_inactive = len(inactive_cpis)
        n_unknown  = cpis['ndvi_mean'].isna().sum()

        print(f"Total CPIS: {n_total}")
        print(f"  Active   (NDVI > {NDVI_THRESHOLD}):  {n_active:5d} ({100*n_active/n_total:.1f}%)")
        print(f"  Inactive (NDVI <= {NDVI_THRESHOLD}): {n_inactive:5d} ({100*n_inactive/n_total:.1f}%)")
        print(f"  No data:                {n_unknown:5d} ({100*n_unknown/n_total:.1f}%)")

        os.makedirs(os.path.dirname(active_path), exist_ok=True)
        os.makedirs(os.path.dirname(inactive_path), exist_ok=True)

        active_cpis.to_file(active_path)
        inactive_cpis.to_file(inactive_path)
        ndvi_df.to_csv(stats_path)

        print(f"\nSaved active CPIS:   {active_path}")
        print(f"Saved inactive CPIS: {inactive_path}")
        print(f"Saved NDVI stats:    {stats_path}")
else:
    print("[Skipped] Requires NDVI extraction from previous cell.")

Active/inactive shapefiles already exist, skipping classification.
  C:/Users/ermil/Documents/Africa_Irrigation\Data/Processed/Active_CPIS-shp/Active_CPIS.shp
  C:/Users/ermil/Documents/Africa_Irrigation\Data/Processed/Inactive_CPIS-shp/Inactive_CPIS.shp
  Delete these files to re-classify (e.g. with a different NDVI_THRESHOLD).


## Visualize NDWI and CPIS Activity

Two panels:
- **Left:** NDWI raster with CPIS outlines
- **Right:** Active (green) vs inactive (red) CPIS on SSA boundaries

In [ ]:
# --- Visualization ---
# Minimum cells to run before this one: ndwi-imports, ndwi-check.
# Everything else is loaded from disk here.
if DATA_AVAILABLE:
    import matplotlib.patches as mpatches
    from rasterio.enums import Resampling as _Resampling
    from shapely.ops import unary_union

    # --- Load all required data from disk ---
    cpis = gpd.read_file(resolve_path(config['SSA_Combined_CPIS_All_shp_path']))

    af_bnds = gpd.read_file(resolve_path(config['Africa_boundaries_shp_path']))
    af_bnds['geometry'] = af_bnds['geometry'].simplify(1000)  # EPSG:3857, meters

    arid_ssa = gpd.read_file(resolve_path(config['SSA_All_by_Country_shp_path']))
    arid_ssa['geometry'] = arid_ssa['geometry'].simplify(0.01)  # EPSG:4326, degrees
    arid_ssa = arid_ssa.to_crs(af_bnds.crs)

    active_cpis   = gpd.read_file(resolve_path(config['Active_CPIS_shp_path']))
    inactive_cpis = gpd.read_file(resolve_path(config['Inactive_CPIS_shp_path']))
    for gdf in [active_cpis, inactive_cpis]:
        if gdf.crs != af_bnds.crs:
            gdf.to_crs(af_bnds.crs, inplace=True)

    print(f"Active CPIS:   {len(active_cpis):,}")
    print(f"Inactive CPIS: {len(inactive_cpis):,}")

    arid_outline = gpd.GeoDataFrame(
        geometry=[unary_union(arid_ssa.geometry)], crs=arid_ssa.crs
    )

    fig, axes = plt.subplots(1, 2, figsize=(28, 14))
    plt.subplots_adjust(wspace=0.05, left=0.05, right=0.95, top=0.9, bottom=0.05)

    # --- Panel 1: NDVI raster with CPIS outlines ---
    # Green = high NDVI = actively irrigated crops; Red = low NDVI = bare soil
    ax = axes[0]
    norm_ndvi = TwoSlopeNorm(vmin=-0.1, vcenter=0.2, vmax=0.7)
    DISPLAY_HEIGHT = 2048
    with rasterio.open(ndvi_vrt_path) as src:
        scale = DISPLAY_HEIGHT / src.height
        display_w = max(1, int(src.width * scale))
        display_h = max(1, int(src.height * scale))
        ndvi_arr = src.read(
            1,
            out_shape=(display_h, display_w),
            resampling=_Resampling.average,
            masked=True,
        )
        raster_crs = src.crs
        extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]

    cpis_raster = cpis.to_crs(raster_crs) if cpis.crs != raster_crs else cpis
    arid_raster = arid_outline.to_crs(raster_crs)

    ax.imshow(ndvi_arr, cmap='RdYlGn', norm=norm_ndvi, alpha=0.85,
              extent=extent, origin='upper', aspect='equal')
    cpis_raster.boundary.plot(ax=ax, color='black', linewidth=0.3, alpha=0.6)
    arid_raster.boundary.plot(ax=ax, color='gray', linewidth=1.2, linestyle='--')
    ax.set_title('NDVI — Green=actively irrigated, Red=bare soil', fontsize=20)
    ax.set_axis_off()

    # --- Panel 2: Active vs inactive CPIS ---
    ax2 = axes[1]
    af_bnds.boundary.plot(ax=ax2, color='dimgray', linewidth=0.5)
    arid_outline.boundary.plot(ax=ax2, color='black', linewidth=1.2, linestyle='--')
    inactive_cpis.plot(ax=ax2, color='#d73027', markersize=3, alpha=0.6)
    active_cpis.plot(ax=ax2, color='#1a9850', markersize=3, alpha=0.6)
    ax2.set_title('Active vs Inactive CPIS (NDVI threshold = 0.2)', fontsize=20)
    ax2.set_axis_off()

    legend_handles = [
        mpatches.Patch(color='#1a9850', label=f'Active NDVI > 0.2  ({len(active_cpis):,})'),
        mpatches.Patch(color='#d73027', label=f'Inactive NDVI \u2264 0.2 ({len(inactive_cpis):,})'),
        mpatches.Patch(edgecolor='black', facecolor='none', linestyle='--',
                       label='Arid region boundary'),
    ]
    ax2.legend(handles=legend_handles, fontsize=14, loc='lower left', framealpha=0.9)

    fig.suptitle('CPIS Activity Classification from Sentinel-2 NDVI (2021)',
                 fontsize=26, fontweight='bold', y=0.95)
    plt.show()
else:
    print("[PLACEHOLDER] Visualization requires Sentinel-2 NDVI data.")
    print("Download data following the instructions in the 'Data Acquisition' section above.")